# blockwise-ilastik-multicut

End-to-end pipeline:
1. Inspect an ilastik `.ilp` project file (possibly trained on multiple sub-volume crops)
2. Re-fit a sklearn RandomForestClassifier on all stored training crops
3. Run blockwise multicut on the full (large) volume

See `README.md` for requirements and background.

In [ ]:
# --- Paths — edit these ---
ILP_PATH = "/path/to/your_project.ilp"
RF_SAVE_PATH = "rf.pkl"

# Map ilastik channel names → (file_path, h5_key_or_None).
# Run the 'Inspect' cell below first to see the exact channel names in your project.
CHANNEL_FILES = {
    "Membrane Probabilities 0": ("/path/to/boundary.h5", "/data"),
    "Raw Data 0":               ("/path/to/raw.h5",      "/data"),
}

# --- Output ---
# For in-memory mode (moderate volumes):
OUTPUT_H5   = "segmentation.h5"
OUTPUT_KEY  = "/seg"
# For lazy / large-volume mode:
OUTPUT_ZARR = "segmentation.zarr"
WS_TMP      = "/tmp/ws_tmp.dat"  # disk space ≈ vol_shape × 8 bytes

# --- Segmentation parameters ---
BETA        = 0.5      # boundary bias: <0.5 → more merges, >0.5 → more splits
BLOCK_SHAPE = (256, 256, 256)
HALO        = (32, 32, 32)
N_THREADS   = 8
USE_2DWS    = False    # set True for strongly anisotropic (2D-stack) data
LAZY        = True     # set False if your data fits comfortably in RAM

## 1. Inspect the .ilp file

In [ ]:
from ilp_reader import discover_lanes, read_feature_names, read_edge_labels

lanes = discover_lanes(ILP_PATH)
print(f"Lanes with training labels: {lanes}")

feature_names = read_feature_names(ILP_PATH)
print("\nChannels and selected features:")
for channel, feats in feature_names.items():
    print(f"  {channel!r}: {feats}")

In [ ]:
import numpy as np

for lane in lanes:
    labels = read_edge_labels(ILP_PATH, lane=lane)
    arr = np.array(list(labels.values()), dtype=np.uint8)
    print(f"Lane {lane}: {len(arr)} annotated edges "
          f"({(arr==1).sum()} merge, {(arr==2).sum()} split)")

## 2. Re-fit sklearn RF on all training lanes

In [ ]:
from fit_classifier import fit_rf_from_ilp
import pickle

# lane=None → automatically reads and concatenates all lanes
rf = fit_rf_from_ilp(
    ilp_path=ILP_PATH,
    lane=None,
    n_estimators=200,
    n_jobs=N_THREADS,
    random_state=42,
    verbose=True,
)

with open(RF_SAVE_PATH, "wb") as f:
    pickle.dump(rf, f)
print(f"Saved classifier to {RF_SAVE_PATH}")

In [ ]:
# --- Feature importances across all lanes ---
import pandas as pd
from ilp_reader import read_training_data

_, _, feature_cols = read_training_data(ILP_PATH, lane=None)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(15).plot.barh(figsize=(8, 5), title="Top-15 feature importances (all crops)");

## 3. Run blockwise multicut on the full volume

With `LAZY=True` the pipeline:
- Opens input channels lazily (zarr / HDF5 datasets, not loaded into memory)
- Runs `blockwise_two_pass_watershed` block-by-block, writing labels to a
  numpy memmap on disk (bounded RAM per block)
- Computes ilastikrag features block-by-block, accumulating edge costs in a
  global dict in RAM
- Runs elf's `blockwise_multicut` (reads watershed from memmap block-by-block)
- Projects node labels to pixels blockwise, writing zarr output

In [ ]:
from multicut_from_ilp import run_blockwise_multicut

channel_specs = [
    f"{name}:{path}:{key}" if key else f"{name}:{path}"
    for name, (path, key) in CHANNEL_FILES.items()
]

run_blockwise_multicut(
    ilp_path=ILP_PATH,
    rf_path=RF_SAVE_PATH,
    channel_specs=channel_specs,
    # lazy mode (large volumes):
    lazy=LAZY,
    output_zarr_path=OUTPUT_ZARR if LAZY else None,
    ws_tmp_path=WS_TMP,
    # in-memory mode (small volumes):
    output_path=None if LAZY else OUTPUT_H5,
    output_key=OUTPUT_KEY,
    beta=BETA,
    block_shape=BLOCK_SHAPE,
    halo=HALO,
    n_threads=N_THREADS,
    use_2dws=USE_2DWS,
    verbose=True,
)

In [ ]:
# --- Quick visual check (middle slice) ---
import matplotlib.pyplot as plt
import zarr, h5py, numpy as np

# Load first channel for overlay
first_name, (first_path, first_key) = next(iter(CHANNEL_FILES.items()))
if first_path.endswith((".h5", ".hdf5")):
    with h5py.File(first_path) as f:
        raw = f[first_key][()]
else:
    raw = zarr.open(first_path, mode="r")[first_key][:] if first_key else zarr.open(first_path)[...]

if LAZY:
    seg = zarr.open(OUTPUT_ZARR, mode="r")[:]
else:
    with h5py.File(OUTPUT_H5) as f:
        seg = f[OUTPUT_KEY][()]

z = raw.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(raw[z], cmap="gray"); axes[0].set_title(f"Input (z={z})")
axes[1].imshow(seg[z], cmap="tab20"); axes[1].set_title("Segmentation")
plt.tight_layout()